<a href="https://colab.research.google.com/github/GarvLamba1/IEEE-GeoPulse-Project/blob/main/IEEE_Pytorch_and_TensorFlow.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import re
import csv

# Define the input file path here
INPUT_XML_FILE = "sensors_data.xml"
OUTPUT_DATASET_FILE = "renewable_energy_dataset.csv"
OUTPUT_EDGE_IMPULSE_TRAIN = "edge_impulse_train.csv"
OUTPUT_EDGE_IMPULSE_INFERENCE = "edge_impulse_inference.csv"

def extract_tag(block: str, tag: str):
    match = re.search(rf"<{tag}>(.*?)</{tag}>", block, re.DOTALL)
    return match.group(1).strip() if match else None

def classify_sample(temperature_c: float, humidity_pct: float, light_raw: float) -> str:
    if temperature_c <= 0 or humidity_pct <= 0 or light_raw < 0:
        return "invalid"
    if not (10 <= temperature_c <= 45):
        return "unsuitable"
    if humidity_pct > 85:
        return "unsuitable"
    if light_raw >= 300:
        return "suitable"
    elif light_raw >= 100:
        return "borderline"
    else:
        return "unsuitable"

def light_band(light_raw: float) -> str:
    if light_raw >= 300:
        return "high"
    elif light_raw >= 100:
        return "medium"
    return "low"

def parse_complete_feeds(xml_text: str):
    feed_blocks = re.findall(r"<feed>(.*?)</feed>", xml_text, re.DOTALL)
    rows = []
    for block in feed_blocks:
        created_at = extract_tag(block, "created-at")
        entry_id = extract_tag(block, "entry-id")
        field1 = extract_tag(block, "field1")
        field2 = extract_tag(block, "field2")
        field3 = extract_tag(block, "field3")
        if None in (created_at, entry_id, field1, field2, field3):
            continue
        try:
            row = {
                "entry_id": int(entry_id),
                "timestamp": created_at,
                "temperature_c": float(field1),
                "humidity_pct": float(field2),
                "light_raw": float(field3),
            }
            rows.append(row)
        except ValueError:
            continue
    dedup = {}
    for row in rows:
        dedup[row["entry_id"]] = row
    rows = list(sorted(dedup.values(), key=lambda x: x["entry_id"]))
    return rows

def main():
    try:
        with open(INPUT_XML_FILE, "r", encoding="utf-8") as f:
            xml_text = f.read()
        rows = parse_complete_feeds(xml_text)
        processed_rows = []
        for row in rows:
            label = classify_sample(row["temperature_c"], row["humidity_pct"], row["light_raw"])
            processed = {**row, "light_band": light_band(row["light_raw"]), "label": label, "install_recommended": 1 if label == "suitable" else 0}
            processed_rows.append(processed)
        with open(OUTPUT_DATASET_FILE, "w", newline="", encoding="utf-8") as f:
            writer = csv.DictWriter(f, fieldnames=["entry_id", "timestamp", "temperature_c", "humidity_pct", "light_raw", "light_band", "label", "install_recommended"])
            writer.writeheader()
            writer.writerows(processed_rows)
        with open(OUTPUT_EDGE_IMPULSE_TRAIN, "w", newline="", encoding="utf-8") as f:
            writer = csv.writer(f)
            writer.writerow(["temperature_c", "humidity_pct", "light_raw", "label"])
            for row in processed_rows:
                if row["label"] != "invalid":
                    writer.writerow([row["temperature_c"], row["humidity_pct"], row["light_raw"], row["label"]])
        with open(OUTPUT_EDGE_IMPULSE_INFERENCE, "w", newline="", encoding="utf-8") as f:
            writer = csv.writer(f)
            writer.writerow(["temperature_c", "humidity_pct", "light_raw"])
            for row in processed_rows:
                if row["label"] != "invalid":
                    writer.writerow([row["temperature_c"], row["humidity_pct"], row["light_raw"]])
        counts = {}
        for row in processed_rows:
            counts[row["label"]] = counts.get(row["label"], 0) + 1
        print("Saved files successfully.")
    except FileNotFoundError:
        print(f"Error: The file {INPUT_XML_FILE} was not found. Please upload it to Colab.")

if __name__ == "__main__":
    main()

Error: The file sensors_data.xml was not found. Please upload it to Colab.


In [3]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix
import tensorflow as tf

DATA_FILE = "renewable_energy_dataset.csv"
MODEL_FILE = "solar_viability_model.h5"


def main():
    df = pd.read_csv(DATA_FILE)

    # Remove invalid rows
    df = df[df["label"] != "invalid"].copy()

    # Features and target
    X = df[["temperature_c", "humidity_pct", "light_raw"]].values
    y_text = df["label"].values

    label_to_id = {
        "unsuitable": 0,
        "borderline": 1,
        "suitable": 2,
    }
    id_to_label = {v: k for k, v in label_to_id.items()}

    y = np.array([label_to_id[label] for label in y_text], dtype=np.int32)

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.2,
        random_state=42,
        stratify=y,
    )

    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)

    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(3,)),
        tf.keras.layers.Dense(16, activation="relu"),
        tf.keras.layers.Dense(8, activation="relu"),
        tf.keras.layers.Dense(3, activation="softmax"),
    ])

    model.compile(
        optimizer="adam",
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )

    early_stop = tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=10,
        restore_best_weights=True,
    )

    model.fit(
        X_train,
        y_train,
        validation_split=0.2,
        epochs=100,
        batch_size=8,
        callbacks=[early_stop],
        verbose=1,
    )

    loss, acc = model.evaluate(X_test, y_test, verbose=0)
    print(f"\nTest accuracy: {acc:.4f}")

    y_pred_probs = model.predict(X_test)
    y_pred = np.argmax(y_pred_probs, axis=1)

    print("\nClassification Report:")
    print(classification_report(
        y_test,
        y_pred,
        target_names=[id_to_label[0], id_to_label[1], id_to_label[2]],
        zero_division=0
    ))

    print("Confusion Matrix:")
    print(confusion_matrix(y_test, y_pred))

    # Save Keras model
    model.save(MODEL_FILE)
    print(f"\nSaved model to {MODEL_FILE}")

    # Save scaler parameters for later inference
    scaler_params = {
        "mean": scaler.mean_.tolist(),
        "scale": scaler.scale_.tolist(),
    }
    print("\nScaler parameters:")
    print(scaler_params)


if __name__ == "__main__":
    main()

Epoch 1/100
7/7 ━━━━━━━━━━━━━━━━━━━━ 1s 39ms/step - accuracy: 0.5542 - loss: 1.0402 - val_accuracy: 0.6923 - val_loss: 0.9841
Epoch 2/100
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.6160 - loss: 1.1310 - val_accuracy: 1.0000 - val_loss: 0.9515
Epoch 3/100
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.8599 - loss: 1.0895 - val_accuracy: 1.0000 - val_loss: 0.9217
Epoch 4/100
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.8742 - loss: 1.0078 - val_accuracy: 1.0000 - val_loss: 0.8903
Epoch 5/100
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.8391 - loss: 1.0630 - val_accuracy: 1.0000 - val_loss: 0.8619
Epoch 6/100
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.9125 - loss: 0.9069 - val_accuracy: 1.0000 - val_loss: 0.8335
Epoch 7/100
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.8716 - loss: 0.9329 - val_accuracy: 1.0000 - val_loss: 0.8035
Epoch 8/100
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.8612 - loss: 0.8906 - val_accuracy: 1.0000 - val_loss:


Classification Report:
              precision    recall  f1-score   support

  unsuitable       0.88      1.00      0.93         7
  borderline       0.00      0.00      0.00         1
    suitable       1.00      1.00      1.00         8

    accuracy                           0.94        16
   macro avg       0.62      0.67      0.64        16
weighted avg       0.88      0.94      0.91        16

Confusion Matrix:
[[7 0 0]
 [1 0 0]
 [0 0 8]]

Saved model to solar_viability_model.h5

Scaler parameters:
{'mean': [24.03125, 20.015625, 232.86457624999997], 'scale': [0.24803918541230538, 4.960759101123033, 228.27296095767412]}


In [4]:
import pandas as pd

INPUT_FILE = "edge_impulse_inference.csv"
OUTPUT_FILE = "threshold_predictions.csv"


def classify_sample(temperature_c: float, humidity_pct: float, light_raw: float) -> str:
    if temperature_c <= 0 or humidity_pct <= 0 or light_raw < 0:
        return "invalid"

    if not (10 <= temperature_c <= 45):
        return "unsuitable"

    if humidity_pct > 85:
        return "unsuitable"

    if light_raw >= 300:
        return "suitable"
    elif light_raw >= 100:
        return "borderline"
    else:
        return "unsuitable"


def install_recommended(label: str) -> int:
    return 1 if label == "suitable" else 0


def main():
    df = pd.read_csv(INPUT_FILE)

    required_cols = {"temperature_c", "humidity_pct", "light_raw"}
    missing = required_cols - set(df.columns)
    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    df["label"] = df.apply(
        lambda row: classify_sample(
            row["temperature_c"],
            row["humidity_pct"],
            row["light_raw"]
        ),
        axis=1
    )
    df["install_recommended"] = df["label"].apply(install_recommended)

    df.to_csv(OUTPUT_FILE, index=False)
    print(f"Saved predictions to {OUTPUT_FILE}")
    print(df.head(20))


if __name__ == "__main__":
    main()

Saved predictions to threshold_predictions.csv
    temperature_c  humidity_pct  light_raw       label  install_recommended
0            24.0          18.0  575.00000    suitable                    1
1            24.0          19.0  565.83331    suitable                    1
2            24.0          19.0  571.66663    suitable                    1
3            24.0          19.0  572.50000    suitable                    1
4            24.0          40.0  570.00000    suitable                    1
5            24.0          42.0  570.83331    suitable                    1
6            24.0          19.0  574.16663    suitable                    1
7            24.0          18.0  570.00000    suitable                    1
8            24.0          18.0  573.33331    suitable                    1
9            24.0          19.0  571.66663    suitable                    1
10           24.0          18.0  570.83331    suitable                    1
11           24.0          19.0  580.0000